# The Second Order Problem
## Olist Brazilian E-Commerce | 99,441 Orders | 9 Linked Tables | SQL

## Executive Summary

This analysis asks why most customers on Olist, a Brazilian e-commerce marketplace, never 
place a second order. Every answer is written in SQL across nine linked tables.

The short answer: it is not the first order. Only 2.16% of customers ever came back, yet 
repeat and one-time buyers had nearly identical first orders, with early deliveries, similar 
review scores and similar costs. Olist is getting the first purchase right. What keeps 
customers from returning lies outside what this data captures, and identifying it is the 
first step to fixing it.

### Key Findings

1. **Retention is the core problem.** Only 2.16% of customers placed a second order, and 
fewer than 1% of any monthly cohort bought again in any later month. Revenue grew from 112k 
in January 2017 to a 988k peak in November 2017, then plateaued through 2018, with growth 
depending almost entirely on new customers.

2. **The first order does not explain it.** Repeat and one-time buyers had similar first 
orders: late delivery rates of 5.0% and 6.7%, average reviews of 4.3 and 4.2, and freight 
shares of 31.6% and 30.8%.

3. **Late deliveries hurt reviews, but they are rare.** Average review scores fall from 4.29 
for early deliveries to 1.70 for orders more than 7 days late, but only about 7% of orders 
arrived late.

4. **Customers who return take their time.** The median repeat customer placed their second 
order 74 days after the first, and 55% of second orders came within 90 days.

### Recommended Actions

- Make cohort retention a headline metric alongside revenue, since revenue growth in 2017 
masked near-zero repeat purchasing.
- Collect post-purchase data, such as marketing contact, discounts and browsing, to diagnose 
the cause.
- Test re-engagement within 90 days of a first order, when 55% of second orders occur, 
starting with the 14,573 one-time, recent, high spend customers, the most valuable group 
yet to return.

## 1. Setup

Loading the nine Olist tables into an in-memory DuckDB database, so every analysis in this notebook can be written in SQL.

In [1]:
# DuckDB is not pre-installed on Kaggle
!pip install duckdb -q

import duckdb
import pandas as pd

con = duckdb.connect()
print("DuckDB ready")

DuckDB ready


In [2]:
# Load all 9 Olist tables into DuckDB directly from CSV
tables = {
    "orders":           "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv",
    "order_items":      "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv",
    "order_payments":   "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv",
    "order_reviews":    "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv",
    "customers":        "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv",
    "products":         "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv",
    "sellers":          "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv",
    "geolocation":      "/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv",
    "categories":       "/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv",
}

for name, path in tables.items():
    con.sql(f"CREATE TABLE {name} AS SELECT * FROM read_csv_auto('{path}')")
    print(f"Loaded: {name}")

Loaded: orders
Loaded: order_items
Loaded: order_payments
Loaded: order_reviews
Loaded: customers
Loaded: products
Loaded: sellers
Loaded: geolocation
Loaded: categories


## 2. Data Quality Checks
Verifying the data before analysis: row counts, unique customers, order status, date ranges, orphan keys and duplicates.

In [3]:
con.sql("SHOW TABLES").df()

,name
0,categories
1,customers
2,geolocation
3,order_items
4,order_payments
5,order_reviews
6,orders
7,products
8,sellers


In [4]:
con.sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
    UNION ALL
    SELECT 'order_items' AS table_name, COUNT(*) AS row_count FROM order_items
    UNION ALL
    SELECT 'order_payments' AS table_name, COUNT(*) AS row_count FROM order_payments
    UNION ALL
    SELECT 'order_reviews' AS table_name, COUNT(*) AS row_count FROM order_reviews
    UNION ALL
    SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
    UNION ALL
    SELECT 'products' AS table_name, COUNT(*) AS row_count FROM products
    UNION ALL
    SELECT 'sellers' AS table_name, COUNT(*) AS row_count FROM sellers
    UNION ALL
    SELECT 'geolocation' AS table_name, COUNT(*) AS row_count FROM geolocation
    UNION ALL
    SELECT 'categories' AS table_name, COUNT(*) AS row_count FROM categories
    ORDER BY row_count DESC
""").df()

,table_name,row_count
0,geolocation,1000163
1,order_items,112650
2,order_payments,103886
3,customers,99441
4,orders,99441
5,order_reviews,99224
6,products,32951
7,sellers,3095
8,categories,71


In [5]:
# customer_id is unique per order, not per person, so count real people via customer_unique_id
con.sql("""
    SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers
    FROM customers
""").df()

,unique_customers
0,96096


In [6]:
# Revenue, delivery and review analysis will use delivered orders only
con.sql("""
    SELECT order_status, COUNT(*) AS orders
    FROM orders
    GROUP BY order_status
    ORDER BY orders DESC
""").df()

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


**Analysis rule:** all revenue, delivery time and review analysis uses delivered orders only (96,478 of 99,441 total).

In [7]:
con.sql("""
    SELECT MIN(order_purchase_timestamp) AS first_order,
           MAX(order_purchase_timestamp) AS last_order
    FROM orders
""").df()

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


**Date range:** Sep 2016 to Oct 2018. First and last months are incomplete and will be excluded from trend analysis.

In [8]:
con.sql("""
    SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
           COUNT(*) AS orders
    FROM orders
    GROUP BY DATE_TRUNC('month', order_purchase_timestamp)
    ORDER BY month
""").df()

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


**Analysis window:** Orders run continuously from January 2017 to August 2018, which is 
the window used for trend and cohort analysis. Outside it the data is sparse: September 
2016 has 4 orders, October 2016 has 324, November 2016 has none and December 2016 has 1. 
After August 2018, September has 16 orders and October has 4. Including any of these 
months would show false drops in monthly trends. Customer-level analysis uses all 
delivered orders, regardless of date.

In [9]:
# Orphan checks: every foreign key should resolve to a row in its parent table
con.sql("""
    SELECT 'order_items -> orders' AS check_name, COUNT(*) AS orphans
    FROM order_items WHERE order_id NOT IN (SELECT order_id FROM orders)
    UNION ALL
    SELECT 'order_payments -> orders', COUNT(*)
    FROM order_payments WHERE order_id NOT IN (SELECT order_id FROM orders)
    UNION ALL
    SELECT 'order_reviews -> orders', COUNT(*)
    FROM order_reviews WHERE order_id NOT IN (SELECT order_id FROM orders)
    UNION ALL
    SELECT 'orders -> customers', COUNT(*)
    FROM orders WHERE customer_id NOT IN (SELECT customer_id FROM customers)
    UNION ALL
    SELECT 'order_items -> products', COUNT(*)
    FROM order_items WHERE product_id NOT IN (SELECT product_id FROM products)
    UNION ALL
    SELECT 'order_items -> sellers', COUNT(*)
    FROM order_items WHERE seller_id NOT IN (SELECT seller_id FROM sellers)
    UNION ALL
    SELECT 'products -> categories', COUNT(*)
    FROM products WHERE product_category_name NOT IN (SELECT product_category_name FROM categories)
""").df()

,check_name,orphans
0,order_items -> orders,0
1,order_payments -> orders,0
2,order_reviews -> orders,0
3,orders -> customers,0
4,order_items -> products,0
5,order_items -> sellers,0
6,products -> categories,13


Six of the seven foreign keys resolve to a valid parent row, so joins across those tables 
lose no data. The exception is products to categories: 13 products have a category name 
with no English translation.

In [10]:
# Grain check: confirm primary keys are unique in lookup tables used in joins
con.sql("""
    SELECT 'products' AS table_name, COUNT(*) AS total_rows, COUNT(DISTINCT product_id) AS distinct_keys
    FROM products
    UNION ALL
    SELECT 'categories', COUNT(*), COUNT(DISTINCT product_category_name)
    FROM categories
    UNION ALL
    SELECT 'customers', COUNT(*), COUNT(DISTINCT customer_id)
    FROM customers
""").df()

,table_name,total_rows,distinct_keys
0,products,32951,32951
1,categories,71,71
2,customers,99441,99441


Primary keys are unique in all three lookup tables, so joins on these columns produce no row multiplication.

In [11]:
# Grain check: order_id should appear exactly once in orders
con.sql("""
    SELECT order_id, COUNT(*) AS occurrences
    FROM orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
""").df()

,order_id,occurrences


In [12]:
# Grain check: check if there are orders with multiple reviews
con.sql("""
    SELECT order_id, COUNT(*) AS occurrences
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
""").df()

,order_id,occurrences
0,b6e5aa946acc4e29e7069510f28a0bce,2
1,98c977c116f7779360e9fecffd3860b6,2
2,d07ea9203ae0a6ea15a2eacc87b4c0f7,2
3,3fe4dbcdb046a475dbf25463c1ca78bd,2
4,9de38c42ea4d33af19aac6c4d6c4b146,2
...,...,...
542,2b0e07b3ff6d3a2db9405022edc865c2,2
543,80fe5b2fe4869330fabbceea5e5e7c0c,2
544,c0db7d31ace61fc360a3eaa34dd3457c,2
545,5f7252bf13701d879f6fc28e72b71a9b,2


547 orders have more than one review. Review scores will be averaged to one value per order before joining to avoid row duplication.

In [13]:
# Grain check: orders with multiple payment rows (expected for installments and vouchers)
con.sql("""
    SELECT order_id, COUNT(*) AS occurrences
    FROM order_payments
    GROUP BY order_id
    HAVING COUNT(*) > 1
""").df()

,order_id,occurrences
0,5262eaeb971616ffef822379ed91896f,2
1,bedff73fa5a137527ba9314cac93b7b3,2
2,d289934bddf878e1cac62c95c8ee5402,2
3,8ca5bdac5ebe8f2d6fc9171d5ebc906a,9
4,fc6e9a6cd6985d3d454cccfdfdf66931,2
...,...,...
2956,5b26d9049a86fab0a83280e17de710ee,2
2957,6b273a63c1e26cdd0d6c8099218fd1ac,2
2958,4f53691b758958dca780dbbe19dad2ab,2
2959,999f2c3acec30e9b25d21ef56a233e45,2


2,961 orders have multiple payment rows due to installments and vouchers. Payments are not 
used to calculate revenue in this analysis, so this does not affect any results.

In [14]:
# Null check: missing delivery dates cannot be used in delivery time calculations
con.sql("""
    SELECT COUNT(*) AS missing_delivery_dates
    FROM orders
    WHERE order_delivered_customer_date IS NULL
""").df()

,missing_delivery_dates
0,2965


In [15]:
# Breakdown by status to confirm nulls are concentrated in non-delivered orders
con.sql("""
    SELECT order_status, 
           COUNT(*) AS total,
           COUNT(order_delivered_customer_date) AS has_delivery_date,
           COUNT(*) - COUNT(order_delivered_customer_date) AS missing_delivery_date
    FROM orders
    GROUP BY order_status
    ORDER BY total DESC
""").df()

,order_status,total,has_delivery_date,missing_delivery_date
0,delivered,96478,96470,8
1,shipped,1107,0,1107
2,canceled,625,6,619
3,unavailable,609,0,609
4,invoiced,314,0,314
5,processing,301,0,301
6,created,5,0,5
7,approved,2,0,2


2,965 orders are missing a delivery date. Of the 96,478 delivered orders, 8 are missing 
a delivery date and will be excluded from delivery time calculations (0.008%, negligible). 
6 canceled orders show a delivery date, likely refund scenarios or data entry errors.

In [16]:
# Null check: missing prices would silently understate revenue
con.sql("""
    SELECT COUNT(*) AS missing_prices
    FROM order_items
    WHERE price IS NULL
""").df()

,missing_prices
0,0


In [17]:
# Null check: products with no category cannot be classified in revenue analysis
con.sql("""
    SELECT COUNT(*) AS missing_categories
    FROM products
    WHERE product_category_name IS NULL
""").df()

,missing_categories
0,610


In [18]:
# Null check: categories with no English translation will show as NULL in analysis
con.sql("""
    SELECT COUNT(*) AS missing_translations
    FROM categories
    WHERE product_category_name_english IS NULL
""").df()

,missing_translations
0,0


Price has no nulls. 610 products (1.9%) have no category and will be labelled 
'Uncategorized' in analysis. All 71 category names have an English translation.

## Analysis Rules

These rules apply to every query in this notebook, confirmed by the checks above.

1. **Scope:** delivered orders only (`order_status = 'delivered'`, 96,478 orders)
2. **Trend analysis window:** January 2017 to August 2018 for monthly trends and cohort analysis. 2016 data has gaps (November missing entirely) and is excluded from trends. September and October 2018 have under 20 orders combined and are excluded from trends. All delivered orders including 2016 are used for customer-level analysis such as repeat rate, RFM and delivery performance.
3. **Customer identity:** `customer_unique_id` identifies a real person. `customer_id` is order-level and is never used for retention or repeat purchase analysis.
4. **Revenue:** sum of `price` from `order_items`. Freight reported separately. Payment table not used for revenue as it includes installment interest and vouchers that make it differ from product price.
5. **Reviews:** averaged to one score per order before joining (`AVG(review_score) GROUP BY order_id`).
6. **Categories:** products with no category (610) are labelled 'Uncategorized', and the 
13 products whose category has no English translation keep their Portuguese category name. 
Category queries use a LEFT JOIN to the translation table so neither group is dropped.
7. **Geolocation:** not used. `customer_state` and `seller_state` are sufficient for regional analysis.

## 3. Revenue and Growth

Monthly revenue trends, category performance and seller rankings across delivered orders from January 2017 to August 2018.

In [19]:
# Monthly revenue and month-over-month change across delivered orders, Jan 2017 to Aug 2018
monthly_revenue_df = con.sql("""
    WITH revenue_by_month AS (
        SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
               SUM(price) AS revenue
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        WHERE order_status = 'delivered'
          AND order_purchase_timestamp >= '2017-01-01'
          AND order_purchase_timestamp < '2018-09-01'
        GROUP BY DATE_TRUNC('month', order_purchase_timestamp)
    )
    SELECT month,
           revenue,
           revenue - LAG(revenue) OVER (ORDER BY month) AS mom_change
    FROM revenue_by_month
    ORDER BY month
""").df()
monthly_revenue_df

,month,revenue,mom_change
0,2017-01-01,111798.36,NaN
1,2017-02-01,234223.40,122425.04
2,2017-03-01,359198.85,124975.45
3,2017-04-01,340669.68,-18529.17
4,2017-05-01,489338.25,148668.57
5,2017-06-01,421923.37,-67414.88
6,2017-07-01,481604.52,59681.15
7,2017-08-01,554699.70,73095.18
8,2017-09-01,607399.67,52699.97
9,2017-10-01,648247.65,40847.98


Checking what drove the November 2017 peak: daily delivered orders for November 2017.

In [20]:
# Daily delivered orders in November 2017, largest days first
con.sql("""
    SELECT CAST(order_purchase_timestamp AS DATE) AS order_date,
           COUNT(DISTINCT order_id) AS orders
    FROM orders
    WHERE order_status = 'delivered'
      AND order_purchase_timestamp >= '2017-11-01'
      AND order_purchase_timestamp < '2017-12-01'
    GROUP BY order_date
    ORDER BY orders DESC
    LIMIT 5
""").df()

,order_date,orders
0,2017-11-24,1147
1,2017-11-25,487
2,2017-11-27,395
3,2017-11-26,382
4,2017-11-28,372


Revenue grew steadily through 2017 from a low base of 112k in January, peaking at 988k in November, driven by Black Friday (24 Nov 2017 had 1,147 orders, more than double any other day that month). December dropped sharply to 726k as the seasonal spike unwound. Through 2018, monthly revenue plateaued between 826k and 978k with no clear growth trend, suggesting Olist had reached a ceiling in customer acquisition or order frequency by this point.

In [21]:
# Revenue and share by product category across all delivered orders
con.sql("""
    SELECT COALESCE(c.product_category_name_english, p.product_category_name, 'Uncategorized') AS category,
           SUM(price) AS revenue,
           ROUND(SUM(price) / SUM(SUM(price)) OVER () * 100, 2) AS category_share
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN categories c ON p.product_category_name = c.product_category_name
    WHERE order_status = 'delivered'
    GROUP BY COALESCE(c.product_category_name_english, p.product_category_name, 'Uncategorized')
    ORDER BY revenue DESC
""").df()

,category,revenue,category_share
0,health_beauty,1233131.72,9.33
1,watches_gifts,1166176.98,8.82
2,bed_bath_table,1023434.76,7.74
3,sports_leisure,954852.55,7.22
4,computers_accessories,888724.61,6.72
...,...,...,...
69,flowers,1110.04,0.01
70,home_comfort_2,760.27,0.01
71,cds_dvds_musicals,730.00,0.01
72,fashion_childrens_clothes,519.95,0.00


Revenue is distributed across 74 product categories with no single dominant segment. 
Health and beauty leads at 9.33%, followed by watches and gifts at 8.82%, but the top 
five categories together account for roughly 40% of total revenue. This breadth means 
Olist's performance is not dependent on any one category, but also that no single 
category offers an obvious lever for growth.

In [22]:
# Top 20 sellers by revenue across all delivered orders
con.sql("""
    SELECT seller_id,
           SUM(price) AS revenue,
           ROUND(SUM(price) / SUM(SUM(price)) OVER () * 100, 2) AS revenue_share,
           RANK() OVER (ORDER BY SUM(price) DESC) AS rank
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE order_status = 'delivered'
    GROUP BY seller_id
    ORDER BY revenue DESC
    LIMIT 20
""").df()

,seller_id,revenue,revenue_share,rank
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,1.72,1
1,53243585a1d6dc2643021fd1853d8905,217940.44,1.65,2
2,4a3ca9315b744ce9f8e9374361493884,196882.12,1.49,3
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14,1.44,4
4,7c67e1448b00f6e969d365cea6b010ab,186570.05,1.41,5
5,7e93a43ef30c4f03f38b393420bc753a,165981.49,1.26,6
6,da8622b14eb17ae2831f4ac5b9dab84a,159816.87,1.21,7
7,7a67c85e85bb2ce8582c35f2203ad736,139658.69,1.06,8
8,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56,1.05,9
9,955fee9216a65b617aa5c0531780ce60,131836.71,1.00,10


The top seller accounts for just 1.72% of total revenue, and the top 20 combined contribute 
about 21%. Revenue is distributed across thousands of sellers with no individual or cluster 
dominating the platform, confirming Olist operates as a highly fragmented marketplace that 
does not depend on any single seller.

## 4. Retention

Measuring how many customers came back, how long it took, and how retention develops month by month across cohorts.

**Definitions for this section**
- Customer: `customer_unique_id`, delivered orders only
- Repeat customer: a customer with delivered orders on at least two different calendar days (same-day orders count as one visit)
- Repeat rate denominator: all unique customers with at least one delivered order, all dates
- Cohort: month of a customer's first delivered purchase
- Matrix window: January 2017 to August 2018, both for cohort start months and activity months
- Retention in month N: the customer placed at least one delivered order N months after their first purchase month

In [23]:
# Check how many customers placed more than one delivered order on the same calendar day
con.sql("""
    SELECT COUNT(*) AS same_day_customers
    FROM (
        SELECT customer_unique_id, DATE_TRUNC('day', order_purchase_timestamp) AS day
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id, DATE_TRUNC('day', order_purchase_timestamp)
        HAVING COUNT(*) > 1
    ) AS same_day_table
""").df()

,same_day_customers
0,857


857 customers placed more than one delivered order on the same calendar day, likely one shopping trip split across separate checkouts. These are treated as a single visit. A repeat customer is defined as a customer with delivered orders on at least two different calendar days.

In [24]:
# Repeat purchase rate: share of customers who ordered on at least two different days
con.sql("""
    WITH customer_days AS (
        SELECT customer_unique_id,
               COUNT(DISTINCT DATE_TRUNC('day', order_purchase_timestamp)) AS distinct_days
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id
    )
    SELECT COUNT(*) AS total_customers,
           COUNT(CASE WHEN distinct_days > 1 THEN 1 END) AS repeat_customers,
           ROUND(COUNT(CASE WHEN distinct_days > 1 THEN 1 END) / COUNT(*) * 100, 2) AS repeat_rate
    FROM customer_days
""").df()

,total_customers,repeat_customers,repeat_rate
0,93358,2015,2.16


Of 93,358 customers with at least one delivered order, only 2,015 placed a second order on a different day, giving a repeat purchase rate of 2.16%. Nearly 98% of Olist customers never came back. This is the central problem the rest of this analysis investigates.

In [25]:
# Median days between first and second delivered order for repeat customers
con.sql("""
    WITH repeat_customer_orders AS (
        SELECT customer_unique_id,
               DATE_TRUNC('day', order_purchase_timestamp) AS day,
               ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY DATE_TRUNC('day', order_purchase_timestamp)) AS purchase_order
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id, DATE_TRUNC('day', order_purchase_timestamp)
    ),
    gap_bw_first_two_orders AS (
        SELECT a.customer_unique_id,
               a.day AS first_order,
               b.day AS second_order,
               DATEDIFF('day', a.day, b.day) AS order_days_difference
        FROM repeat_customer_orders a
        JOIN repeat_customer_orders b ON a.customer_unique_id = b.customer_unique_id
        WHERE a.purchase_order = 1
          AND b.purchase_order = 2
    )
    SELECT MEDIAN(order_days_difference) AS median_days_to_second_order
    FROM gap_bw_first_two_orders
""").df()

,median_days_to_second_order
0,74.0


In [26]:
# Distribution of days between first and second order for repeat customers
con.sql("""
    WITH repeat_customer_orders AS (
        SELECT customer_unique_id,
               DATE_TRUNC('day', order_purchase_timestamp) AS day,
               ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY DATE_TRUNC('day', order_purchase_timestamp)) AS purchase_order
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id, DATE_TRUNC('day', order_purchase_timestamp)
    ),
    gap_bw_first_two_orders AS (
        SELECT a.customer_unique_id,
               a.day AS first_order,
               b.day AS second_order,
               DATEDIFF('day', a.day, b.day) AS order_days_difference
        FROM repeat_customer_orders a
        JOIN repeat_customer_orders b ON a.customer_unique_id = b.customer_unique_id
        WHERE a.purchase_order = 1
          AND b.purchase_order = 2
    )
    SELECT COUNT(*) AS number_of_customers,
           CASE
               WHEN order_days_difference <= 30 THEN '30 or below'
               WHEN order_days_difference <= 90 THEN '31-90'
               WHEN order_days_difference <= 180 THEN '91-180'
               ELSE 'Over 180'
           END AS days_to_second_order
    FROM gap_bw_first_two_orders
    GROUP BY CASE
               WHEN order_days_difference <= 30 THEN '30 or below'
               WHEN order_days_difference <= 90 THEN '31-90'
               WHEN order_days_difference <= 180 THEN '91-180'
               ELSE 'Over 180'
             END
    ORDER BY MIN(order_days_difference)
""").df()

,number_of_customers,days_to_second_order
0,600,30 or below
1,507,31-90
2,423,91-180
3,485,Over 180


Among the 2,015 repeat customers, the median time to a second order was 74 days. The 
distribution is spread fairly evenly: 30% returned within 30 days, 25% within 31 to 90, 
21% within 91 to 180, and 24% took over 180 days. There is no typical return window, 
which points to weak habitual purchasing on the platform.

In [27]:
# Cohort retention matrix: share of each monthly cohort still purchasing in later months
cohort_df = con.sql("""
    WITH first_purchase_table AS (
        SELECT c.customer_unique_id,
               MIN(DATE_TRUNC('month', order_purchase_timestamp)) AS first_purchase_month
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id
    ),
    order_gap_table AS (
        SELECT fpt.customer_unique_id,
               first_purchase_month,
               DATE_TRUNC('month', order_purchase_timestamp) AS order_month,
               DATEDIFF('month', first_purchase_month, DATE_TRUNC('month', order_purchase_timestamp)) AS month_number
        FROM first_purchase_table fpt
        JOIN customers c ON fpt.customer_unique_id = c.customer_unique_id
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE order_status = 'delivered'
          AND order_purchase_timestamp >= '2017-01-01'
          AND order_purchase_timestamp < '2018-09-01'
        GROUP BY fpt.customer_unique_id, first_purchase_month,
                 DATE_TRUNC('month', order_purchase_timestamp)
    ),
    retained_users_table AS (
        SELECT first_purchase_month,
               month_number,
               COUNT(DISTINCT customer_unique_id) AS retained_customers
        FROM order_gap_table
        WHERE first_purchase_month >= '2017-01-01'
          AND first_purchase_month < '2018-09-01'
        GROUP BY first_purchase_month, month_number
    )
    SELECT first_purchase_month,
           month_number,
           retained_customers,
           FIRST_VALUE(retained_customers) OVER (PARTITION BY first_purchase_month ORDER BY month_number) AS cohort_size,
           ROUND(retained_customers / FIRST_VALUE(retained_customers) OVER (PARTITION BY first_purchase_month ORDER BY month_number) * 100.0, 1) AS retention_rate
    FROM retained_users_table
    ORDER BY first_purchase_month, month_number
""").df()
cohort_df

,first_purchase_month,month_number,retained_customers,cohort_size,retention_rate
0,2017-01-01,0,717,717,100.0
1,2017-01-01,1,2,717,0.3
2,2017-01-01,2,2,717,0.3
3,2017-01-01,3,1,717,0.1
4,2017-01-01,4,3,717,0.4
...,...,...,...,...,...
202,2018-06-01,1,25,5878,0.4
203,2018-06-01,2,16,5878,0.3
204,2018-07-01,0,5949,5949,100.0
205,2018-07-01,1,31,5949,0.5


In [28]:
# Pivot to wide format: rows as cohorts, columns as month numbers
retention_matrix = cohort_df.pivot(
    index='first_purchase_month',
    columns='month_number',
    values='retention_rate'
)
retention_matrix

month_number,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19
first_purchase_month,,,,,,,,,,,,,,,,,,,
2017-01-01,100.0,0.3,0.3,0.1,0.4,0.1,0.4,0.1,0.1,NaN,0.4,0.1,0.7,0.4,0.1,0.1,0.3,0.4,0.1
2017-02-01,100.0,0.2,0.3,0.1,0.4,0.1,0.2,0.2,0.1,0.2,0.1,0.3,0.1,0.2,0.1,0.1,0.1,0.2,NaN
2017-03-01,100.0,0.4,0.4,0.4,0.4,0.2,0.2,0.3,0.3,0.1,0.4,0.1,0.2,0.1,0.2,0.2,0.1,0.1,NaN
2017-04-01,100.0,0.6,0.2,0.2,0.3,0.3,0.4,0.3,0.3,0.2,0.3,0.1,0.0,0.0,0.1,0.1,0.1,NaN,NaN
2017-05-01,100.0,0.5,0.5,0.3,0.3,0.3,0.4,0.1,0.3,0.3,0.3,0.3,0.2,0.0,0.2,0.2,NaN,NaN,NaN
2017-06-01,100.0,0.5,0.4,0.4,0.3,0.4,0.4,0.2,0.1,0.2,0.3,0.4,0.2,0.2,0.2,NaN,NaN,NaN,NaN
2017-07-01,100.0,0.5,0.3,0.2,0.3,0.2,0.3,0.1,0.2,0.3,0.2,0.3,0.1,0.2,NaN,NaN,NaN,NaN,NaN
2017-08-01,100.0,0.7,0.3,0.3,0.3,0.5,0.3,0.3,0.1,0.1,0.2,0.2,0.1,NaN,NaN,NaN,NaN,NaN,NaN
2017-09-01,100.0,0.7,0.5,0.3,0.4,0.2,0.2,0.2,0.3,0.2,0.2,0.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Retention across all cohorts is consistently below 1% in every month after the first purchase. Month 1 retention ranges from 0.2% to 0.7% with no cohort ever exceeding 1% in any subsequent month. There is no improvement over time - later cohorts show the same pattern as earlier ones, suggesting the low repeat rate is structural rather than a temporary issue. Once a customer buys from Olist, they almost never return.

## 5. Customer and Service Analysis

RFM segmentation across all customers, delivery performance against review scores, a 
comparison of the first order experience of one-time and repeat buyers, and freight cost 
by region.

### RFM Segmentation

Customers are scored on recency (days since last purchase, measured from the last order date 
in the dataset) and monetary value (total spend), each split into five equal groups with 
NTILE, where 5 is best. Frequency is not scored this way: 97.8% of customers have exactly one 
purchase day, so NTILE would split identical customers into different groups at random. 
Instead, frequency is a flag: repeat (purchases on two or more days) or one-time. Recency 
scores of 4 to 5 count as recent and 1 to 3 as lapsed; monetary scores of 4 to 5 count as 
high spend and 1 to 3 as low spend, giving eight segments.

In [29]:
# RFM segmentation: customers scored on recency and monetary, frequency as repeat flag
# Ties are broken by customer_unique_id so every run gives the same segments
rfm_customers_df = con.sql("""
    WITH customer_rfm_base AS (
        SELECT customer_unique_id,
               DATEDIFF('day', MAX(DATE_TRUNC('day', order_purchase_timestamp)),
                        (SELECT MAX(order_purchase_timestamp) FROM orders)) AS days_since_last_purchase,
               COUNT(DISTINCT DATE_TRUNC('day', order_purchase_timestamp)) AS no_of_purchase_days,
               ROUND(SUM(price), 2) AS spend
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id
    ),
    ntile_table AS (
        SELECT customer_unique_id,
               days_since_last_purchase,
               spend,
               no_of_purchase_days,
               NTILE(5) OVER (ORDER BY days_since_last_purchase DESC, customer_unique_id) AS recency_score,
               NTILE(5) OVER (ORDER BY spend, customer_unique_id) AS monetary_score,
               CASE WHEN no_of_purchase_days > 1 THEN 1 ELSE 0 END AS frequency_flag
        FROM customer_rfm_base
    )
    SELECT customer_unique_id,
           days_since_last_purchase,
           spend,
           no_of_purchase_days,
           recency_score,
           monetary_score,
           frequency_flag,
           CASE
               WHEN frequency_flag = 1 AND recency_score BETWEEN 4 AND 5 AND monetary_score BETWEEN 4 AND 5 THEN 'Repeat, recent, high spend'
               WHEN frequency_flag = 1 AND recency_score BETWEEN 4 AND 5 AND monetary_score BETWEEN 1 AND 3 THEN 'Repeat, recent, low spend'
               WHEN frequency_flag = 1 AND recency_score BETWEEN 1 AND 3 AND monetary_score BETWEEN 4 AND 5 THEN 'Repeat, lapsed, high spend'
               WHEN frequency_flag = 1 AND recency_score BETWEEN 1 AND 3 AND monetary_score BETWEEN 1 AND 3 THEN 'Repeat, lapsed, low spend'
               WHEN frequency_flag = 0 AND recency_score BETWEEN 4 AND 5 AND monetary_score BETWEEN 4 AND 5 THEN 'One-time, recent, high spend'
               WHEN frequency_flag = 0 AND recency_score BETWEEN 4 AND 5 AND monetary_score BETWEEN 1 AND 3 THEN 'One-time, recent, low spend'
               WHEN frequency_flag = 0 AND recency_score BETWEEN 1 AND 3 AND monetary_score BETWEEN 4 AND 5 THEN 'One-time, lapsed, high spend'
               WHEN frequency_flag = 0 AND recency_score BETWEEN 1 AND 3 AND monetary_score BETWEEN 1 AND 3 THEN 'One-time, lapsed, low spend'
           END AS segment
    FROM ntile_table
""").df()
rfm_customers_df

,customer_unique_id,days_since_last_purchase,spend,no_of_purchase_days,recency_score,monetary_score,frequency_flag,segment
0,b38211bd797f4fdd81a98b9d1754b606,176,0.85,1,4,1,0,"One-time, recent, low spend"
1,d80730c15c647bc8f2ad77c908ba5ca9,176,0.85,1,4,1,0,"One-time, recent, low spend"
2,317cfc692e3f86c45c95697c61c853a6,53,2.20,1,5,1,0,"One-time, recent, low spend"
3,bd06ce0e06ad77a7f681f1a4960a3cc6,399,2.29,1,2,1,0,"One-time, lapsed, low spend"
4,cf3839da0d9492ad151690b65f45d800,257,2.99,1,3,1,0,"One-time, lapsed, low spend"
...,...,...,...,...,...,...,...,...
93353,459bef486812aa25204be022145caa62,84,6729.00,1,5,5,0,"One-time, recent, high spend"
93354,dc4802a71eae9be1dd28f5d788ceb526,612,6735.00,1,1,5,0,"One-time, lapsed, high spend"
93355,763c8b1c9c68a0229c42c9fc6f662b93,94,7160.00,1,5,5,0,"One-time, recent, high spend"
93356,da122df9eeddfedc1dc1f5349a1a690c,564,7388.00,1,1,5,0,"One-time, lapsed, high spend"


In [30]:
# Segment distribution: customer count per RFM segment
con.sql("""
    SELECT segment, COUNT(*) AS customers
    FROM rfm_customers_df
    GROUP BY segment
    ORDER BY customers DESC
""").df()

,segment,customers
0,"One-time, lapsed, low spend",33872
1,"One-time, recent, low spend",21709
2,"One-time, lapsed, high spend",21189
3,"One-time, recent, high spend",14573
4,"Repeat, recent, high spend",837
5,"Repeat, lapsed, high spend",743
6,"Repeat, recent, low spend",223
7,"Repeat, lapsed, low spend",212


The dominant segment is one-time, lapsed, low spend at 33,872 customers (36%), 
confirming that most of Olist's customer base has disengaged without ever spending 
much. The next two largest segments, one-time, recent, low spend (21,709) 
and one-time, lapsed, high spend (21,189), point to two different problems: customers 
who are still active but spending little, and high-value customers who have not returned. 
One-time, recent, high spend at 14,573 is the most actionable one-time segment, 
representing customers worth converting to repeat buyers before they lapse.

Among repeat buyers, the 2,015 total matches the repeat rate from Section 4 exactly. 
Repeat, recent, high spend leads at 837, followed by repeat, lapsed, high spend at 
743, a group worth targeting for win-back given their demonstrated spend. Repeat, low 
spend segments are small (223 and 212) but not negligible. The priority is clear: 
retain and re-engage the high-spend segments, and convert one-time recent buyers 
before they join the lapsed majority.

In [31]:
# Delivery delay vs review score: does late delivery hurt customer satisfaction?
delivery_reviews_df = con.sql("""
    WITH updated_order_reviews AS (
        SELECT order_id, AVG(review_score) AS avg_review_score
        FROM order_reviews
        GROUP BY order_id
    ),
    delivery_delay AS (
        SELECT o.order_id,
               avg_review_score,
               DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date) AS delay_in_days
        FROM updated_order_reviews uor
        JOIN orders o ON uor.order_id = o.order_id
        WHERE order_status = 'delivered'
          AND order_delivered_customer_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN delay_in_days < 0 THEN 'Early'
            WHEN delay_in_days = 0 THEN 'On time'
            WHEN delay_in_days BETWEEN 1 AND 3 THEN '1 to 3 days late'
            WHEN delay_in_days BETWEEN 4 AND 7 THEN '4 to 7 days late'
            ELSE 'Over 7 days late'
        END AS delivery_time,
        COUNT(*) AS orders,
        ROUND(AVG(avg_review_score), 2) AS avg_review_score
    FROM delivery_delay
    GROUP BY CASE
            WHEN delay_in_days < 0 THEN 'Early'
            WHEN delay_in_days = 0 THEN 'On time'
            WHEN delay_in_days BETWEEN 1 AND 3 THEN '1 to 3 days late'
            WHEN delay_in_days BETWEEN 4 AND 7 THEN '4 to 7 days late'
            ELSE 'Over 7 days late'
        END
    ORDER BY MIN(delay_in_days)
""").df()
delivery_reviews_df

,delivery_time,orders,avg_review_score
0,Early,88163,4.29
1,On time,1280,4.04
2,1 to 3 days late,1852,3.29
3,4 to 7 days late,1748,2.11
4,Over 7 days late,2781,1.70


92% of orders in this analysis arrived before the estimated date, suggesting Olist 
systematically sets conservative delivery estimates. Review scores drop sharply as delay 
increases, from 4.29 for early deliveries to 4.04 for on-time, 3.29 for 1 to 3 days late, 
2.11 for 4 to 7 days late, and 1.70 for orders over 7 days late. Even a small delay is 
associated with a sharp drop in satisfaction. Late delivery is not the only factor behind 
low review scores, but the pattern is consistent enough to treat it as a important one.

In [32]:
# First order experience: comparing repeat buyers vs one-time buyers on their first order
first_order_df = con.sql("""
    WITH first_order AS (
        SELECT customer_unique_id, order_id AS first_order_id
        FROM (
            SELECT c.customer_unique_id,
                   o.order_id,
                   ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp, o.order_id) AS order_rank
            FROM customers c
            JOIN orders o ON c.customer_id = o.customer_id
            WHERE o.order_status = 'delivered'
        ) ranked_orders
        WHERE order_rank = 1
    ),
    order_items_table AS (
        SELECT order_id, SUM(price) AS total_price, SUM(freight_value) AS freight_cost
        FROM order_items
        GROUP BY order_id
    ),
    order_reviews_table AS (
        SELECT order_id, AVG(review_score) AS avg_review_score
        FROM order_reviews
        GROUP BY order_id
    ),
    frequency_flag_table AS (
        SELECT customer_unique_id,
               CASE WHEN COUNT(DISTINCT DATE_TRUNC('day', order_purchase_timestamp)) > 1 
                    THEN 'Repeat' ELSE 'One-time' END AS customer_type
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE order_status = 'delivered'
        GROUP BY customer_unique_id
    ),
    first_order_experience AS (
        SELECT c.customer_unique_id,
               DATEDIFF('day', order_estimated_delivery_date, order_delivered_customer_date) AS delay_in_days,
               avg_review_score,
               total_price,
               freight_cost,
               customer_type
        FROM first_order fo
        JOIN orders o ON fo.first_order_id = o.order_id
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_items_table oit ON o.order_id = oit.order_id
        JOIN order_reviews_table ort ON oit.order_id = ort.order_id
        JOIN frequency_flag_table fft ON c.customer_unique_id = fft.customer_unique_id
        WHERE order_delivered_customer_date IS NOT NULL
    )
    SELECT customer_type,
           ROUND(AVG(delay_in_days), 1) AS average_delay,
           ROUND(COUNT(CASE WHEN delay_in_days > 0 THEN 1 END) / COUNT(*) * 100.0, 1) AS pct_delivered_late,
           ROUND(AVG(avg_review_score), 1) AS average_review_score,
           ROUND(AVG(total_price), 1) AS average_order_value,
           ROUND(AVG(freight_cost / total_price * 100.0), 1) AS avg_freight_pct
    FROM first_order_experience
    GROUP BY customer_type
    ORDER BY customer_type
""").df()
first_order_df

,customer_type,average_delay,pct_delivered_late,average_review_score,average_order_value,avg_freight_pct
0,One-time,-11.9,6.7,4.2,137.5,30.8
1,Repeat,-12.7,5.0,4.3,125.3,31.6


The first order experience of repeat buyers and one-time buyers is nearly identical across 
every dimension measured. Both groups received their first order early on average, had late 
delivery rates under 7%, left average review scores of 4.2 and 4.3, and paid similar freight 
as a share of order value. One-time buyers actually spent slightly more on their first order 
(137.5 vs 125.3). A poor first experience does not explain Olist's 2.16% repeat rate. The 
retention problem is not one of delivery or satisfaction, and it runs deeper than the first 
order.

In [33]:
# Freight cost by region: which states pay the highest freight relative to order value
freight_by_state_df =con.sql("""
    WITH order_items_table AS (
        SELECT order_id,
               SUM(price) AS order_value,
               SUM(freight_value) AS freight_cost
        FROM order_items
        GROUP BY order_id
    )
    SELECT customer_state,
           COUNT(*) AS order_count,
           ROUND(AVG(freight_cost), 2) AS avg_freight_cost,
           ROUND(SUM(freight_cost) / SUM(order_value) * 100.0, 2) AS freight_pct_of_order_value
    FROM order_items_table oit
    JOIN orders o ON oit.order_id = o.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE order_status = 'delivered'
    GROUP BY customer_state
    ORDER BY freight_pct_of_order_value DESC
""").df()
freight_by_state_df

,customer_state,order_count,avg_freight_cost,freight_pct_of_order_value
0,RR,41,48.34,28.08
1,MA,717,42.95,26.32
2,RO,243,46.43,24.70
3,AM,145,37.45,24.51
4,SE,335,40.94,24.24
5,PI,476,42.98,24.15
6,TO,274,42.35,23.98
7,AC,80,45.55,22.88
8,PE,1593,35.83,22.66
9,RN,474,39.26,22.66


Freight as a share of order value ranges from 13.9% to 28.1% across states, and the 
pattern follows order volume. São Paulo (SP), with 40,501 orders and 42% of all delivered 
orders, pays the lowest share at 13.9%. The five highest-volume states all pay under 
18.5%. Every state paying 22% or more has fewer than 1,600 orders, and at the top end, 
customers pay about twice São Paulo's rate. States with fewer than 100 orders (RR, AP 
and AC) should be treated as directional only.

## 6. Conclusions, Recommendations and Limitations

What the findings mean when read together, what Olist should do about them, and where the analysis reaches its limits.

### Conclusions

Olist's core problem is retention. Only 2.16% of customers ever placed a second order, and 
in every monthly cohort fewer than 1% bought again in any later month. With almost no repeat 
purchasing, revenue growth through 2017 depended on acquiring new customers, and when revenue 
plateaued in 2018 there was no repeat base to sustain it. The analysis rules out the first 
order experience as the main explanation: repeat and one-time buyers had nearly identical 
first orders on delivery timing, review scores, order value and freight share, and Olist is 
already getting that first purchase right, with early deliveries and average reviews above 4 
for both groups. Late deliveries are associated with much lower review scores, but only about 
7% of orders arrived late, too few to account for a 98% non-return rate. This tells Olist 
where not to look: faster delivery or better service on the first order is unlikely, on its 
own, to fix retention. Identifying what does drive it will require investigating factors 
this dataset does not capture.

### Recommendations

- **Make retention a headline metric.** Monthly revenue grew through 2017 while repeat 
  purchasing stayed below 1% in every cohort. Tracking cohort retention alongside revenue 
  would have shown the problem much earlier than revenue alone.
- **Collect the data needed to diagnose the problem.** Track post-purchase activity: 
  marketing contact, ad exposure, discounts offered, cross-sell and upsell attempts, and 
  customer browsing after the first order. Without it, the cause of low retention cannot 
  be identified.
- **Test post-purchase re-engagement rather than assume it works.** Contact a randomly 
  selected group of recent one-time buyers and hold back a comparable group, then compare 
  repeat rates. 55% of second orders happen within 90 days of the first, so that is the 
  window to test in.
- **Focus spend on the highest-value customers.** The 14,573 one-time, recent, high spend 
  customers are the natural test group, and the 743 repeat, lapsed, high spend customers 
  are the clearest win-back target. The largest segment, 33,872 one-time, lapsed, low spend 
  customers, is the weakest case for marketing budget.
- **Protect delivery performance.** Late deliveries are rare but are associated with 
  review scores as low as 1.70. Keeping delivery estimates conservative protects 
  satisfaction, even though it is not the main retention lever.

### Limitations

- **No post-purchase data.** The dataset records orders, deliveries and reviews, but nothing 
  about what happens after a purchase, such as marketing contact, discounts or browsing. 
  These are the factors that could explain retention, and they cannot be examined here.
- **Associations, not causes.** The analysis is descriptive. Relationships such as delivery 
  delay and review score are associations, and no statistical significance tests were run, 
  so small differences, such as those between repeat and one-time buyers' first orders, 
  should not be read as definitive.
- **The repeat rate is likely understated.** Customers who first bought in mid-2018 had only 
  weeks to return before the data ends, while the median repeat customer took 74 days. 
  The cohort matrix accounts for this; the overall 2.16% rate does not. Customers are also 
  identified by account, so anyone who ordered again under a different account counts as 
  a new customer.
- **Historical data.** The data covers 2016 to 2018 on a single marketplace and may not 
  reflect Olist's current performance.

## 7. Summary Tables for Dashboards

Exporting the query results that feed the Tableau dashboard as CSV files.

In [34]:
import os

# Create output folder for dashboard exports
os.makedirs("dashboard_exports", exist_ok=True)

# Export all five tables as CSVs for Tableau
monthly_revenue_df.to_csv("dashboard_exports/monthly_revenue.csv", index=False)
cohort_df.to_csv("dashboard_exports/cohort_retention.csv", index=False)
rfm_customers_df.to_csv("dashboard_exports/rfm_customers.csv", index=False)
delivery_reviews_df.to_csv("dashboard_exports/delivery_vs_reviews.csv", index=False)
first_order_df.to_csv("dashboard_exports/first_order_experience.csv", index=False)

print("All five exports complete.")

All five exports complete.
